# Week 3 — Data Preprocessing

## 1. Missing-Data Assessment

Analyze missing pollutant readings in Delhi before selecting appropriate preprocessing methods.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/raw/city_day.csv")

delhi_df = df[df["City"] == "Delhi"].copy()
delhi_df["Date"] = pd.to_datetime(delhi_df["Date"])

delhi_df = delhi_df.sort_values("Date")
delhi_df = delhi_df.set_index("Date").asfreq("D")

print("Delhi dataset shape:", delhi_df.shape)

Delhi dataset shape: (2009, 15)


In [2]:
missing_summary = pd.DataFrame({
    "Missing Count": delhi_df.isna().sum(),
    "Missing %": (delhi_df.isna().mean() * 100).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing %", ascending=False
)

display(missing_summary)

,Missing Count,Missing %
Xylene,781,38.88
SO2,110,5.48
O3,84,4.18
PM10,77,3.83
AQI,10,0.50
AQI_Bucket,10,0.50
NH3,9,0.45
PM2.5,2,0.10
NO2,2,0.10
NO,2,0.10


## 2. Missing-Value Gap Analysis

Examine consecutive missing-value periods to determine whether forward filling is appropriate.

In [3]:
def longest_missing_gap(series):
    missing = series.isna()
    groups = (~missing).cumsum()
    return missing.groupby(groups).sum().max()

gap_summary = pd.DataFrame({
    "Missing Count": delhi_df.isna().sum(),
    "Longest Missing Gap": [
        longest_missing_gap(delhi_df[col])
        for col in delhi_df.columns
    ]
})

display(gap_summary.sort_values("Longest Missing Gap", ascending=False))

,Missing Count,Longest Missing Gap
Xylene,781,606
SO2,110,81
O3,84,81
PM10,77,58
AQI,10,4
AQI_Bucket,10,4
PM2.5,2,2
NO,2,2
NH3,9,2
NO2,2,2


In [4]:
print("Dates with missing PM2.5:")
display(delhi_df[delhi_df["PM2.5"].isna()][["PM2.5", "AQI", "AQI_Bucket"]])

Dates with missing PM2.5:


,PM2.5,AQI,AQI_Bucket
Date,,,
2017-08-12,NaN,45.0,Good
2017-08-13,NaN,NaN,NaN


## 3. Proposed Missing-Data Treatment Policy

- Exclude Xylene due to high missingness (38.88%) and a maximum consecutive gap of 606 days.
- Forward-fill short gaps (maximum 2 consecutive days) in selected pollutant predictors.
- Preserve longer missing periods instead of fabricating extended observations.
- Preserve missing PM2.5 and AQI_Bucket values because these columns are used to construct prediction targets.
- Use only previously observed values for filling to prevent look-ahead leakage.

**Status:** Standard preprocessing policy adopted for Week 3.

All subsequent feature-engineering tasks will use this preprocessing pipeline to maintain consistency across the project.

In [5]:
pollutant_cols = ["SO2", "O3", "PM10", "NO", "NO2", "NH3"]

for col in pollutant_cols:
    missing = delhi_df[col].isna()
    gap_id = missing.ne(missing.shift()).cumsum()
    gap_length = missing.groupby(gap_id).transform("sum")

    eligible = missing & (gap_length <= 2)

    print(f"{col}: {eligible.sum()} missing days in short gaps")

SO2: 1 missing days in short gaps
O3: 3 missing days in short gaps
PM10: 9 missing days in short gaps
NO: 2 missing days in short gaps
NO2: 2 missing days in short gaps
NH3: 9 missing days in short gaps


## 4. Apply Missing-Data Treatment

Create a separate processed DataFrame, exclude Xylene, and forward-fill only short missing gaps in selected pollutant predictors.

In [6]:
processed_df = delhi_df.drop(columns=["Xylene"]).copy()

for col in pollutant_cols:
    missing = processed_df[col].isna()

    gap_id = missing.ne(missing.shift()).cumsum()
    gap_length = missing.groupby(gap_id).transform("sum")

    eligible = missing & (gap_length <= 2)

    # Only use previously observed values
    previous_value = processed_df[col].ffill()

    # Fill only eligible gaps
    processed_df.loc[eligible, col] = previous_value[eligible]

print("Processed dataset shape:", processed_df.shape)

Processed dataset shape: (2009, 14)


In [7]:
comparison = pd.DataFrame({
    "Before": delhi_df.isna().sum(),
    "After": processed_df.isna().sum()
})

comparison["Filled"] = comparison["Before"] - comparison["After"]

display(comparison)

,Before,After,Filled
AQI,10,10.0,0.0
AQI_Bucket,10,10.0,0.0
Benzene,0,0.0,0.0
CO,0,0.0,0.0
City,0,0.0,0.0
NH3,9,0.0,9.0
NO,2,0.0,2.0
NO2,2,0.0,2.0
NOx,0,0.0,0.0
O3,84,81.0,3.0


## 5. Preprocessing Validation

Verify that original non-missing measurements remain unchanged and identify remaining missing values.

In [8]:
# Verify original non-missing values remain unchanged
original_values = delhi_df[processed_df.columns]

unchanged = (
    processed_df.where(original_values.notna())
    .equals(original_values.where(original_values.notna()))
)

print("Original valid measurements preserved:", unchanged)

print("PM2.5 unchanged:",
      processed_df["PM2.5"].equals(delhi_df["PM2.5"]))

print("AQI_Bucket unchanged:",
      processed_df["AQI_Bucket"].equals(delhi_df["AQI_Bucket"]))

Original valid measurements preserved: True
PM2.5 unchanged: True
AQI_Bucket unchanged: True


In [9]:
remaining_missing = processed_df.isna().sum()

print("Remaining missing values:")
print(remaining_missing[remaining_missing > 0])

print("\nTotal remaining missing values:",
      remaining_missing.sum())

Remaining missing values:
PM2.5           2
PM10           68
SO2           109
O3             81
AQI            10
AQI_Bucket     10
dtype: int64

Total remaining missing values: 280


## 6. Preprocessing Summary

- Excluded Xylene due to 38.88% missingness and a maximum missing gap of 606 days.
- Filled 26 missing pollutant readings belonging to gaps of at most two consecutive days, using previously available measurements.
- Preserved longer missing periods in PM10, SO2, and O3.
- Preserved missing PM2.5, AQI, and AQI_Bucket values.
- Verified that all original non-missing measurements remain unchanged.
- The processed dataset contains 2,009 rows and 14 columns, with 280 missing values remaining.

**Status:** Standard preprocessing policy adopted for Week 3.

All subsequent feature-engineering tasks will use this preprocessing pipeline to maintain consistency across the project.

**Limitation:** Remaining missing predictor values must be handled consistently during model preparation. Forward filling does not use future readings, but further validation is required to ensure no train/test leakage.

## 7. Pollutant Lag Features — Deepanshu

Lag features use historical pollutant measurements to help predict next-day air quality. For each selected pollutant, we create lag features for 1, 2, 3, and 7 days before the current date. The dataset is maintained on a continuous daily calendar so that each lag corresponds to the correct number of days. Missing values are preserved rather than filled using future observations.

In [10]:
# Pollutant lag features — Deepanshu

processed_df = processed_df.sort_index().copy()

pollutants_for_lags = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"
]

# Keep only pollutant columns that exist in the dataset
pollutants_for_lags = [
    col for col in pollutants_for_lags
    if col in processed_df.columns
]

lag_periods = [1, 2, 3, 7]

for pollutant in pollutants_for_lags:
    for lag in lag_periods:
        processed_df[f"{pollutant}_lag_{lag}"] = (
            processed_df[pollutant].shift(lag)
        )

print("Lag features created successfully.")
print("Dataset shape:", processed_df.shape)

lag_columns = [
    f"{pollutant}_lag_{lag}"
    for pollutant in pollutants_for_lags
    for lag in lag_periods
]

display(processed_df[lag_columns].head(10))

Lag features created successfully.
Dataset shape: (2009, 58)


,PM2.5_lag_1,PM2.5_lag_2,PM2.5_lag_3,PM2.5_lag_7,PM10_lag_1,PM10_lag_2,PM10_lag_3,PM10_lag_7,NO_lag_1,NO_lag_2,...,O3_lag_3,O3_lag_7,Benzene_lag_1,Benzene_lag_2,Benzene_lag_3,Benzene_lag_7,Toluene_lag_1,Toluene_lag_2,Toluene_lag_3,Toluene_lag_7
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,313.22,NaN,NaN,NaN,607.98,NaN,NaN,NaN,69.16,NaN,...,NaN,NaN,14.36,NaN,NaN,NaN,24.86,NaN,NaN,NaN
2015-01-03,186.18,313.22,NaN,NaN,269.55,607.98,NaN,NaN,62.09,69.16,...,NaN,NaN,10.55,14.36,NaN,NaN,20.09,24.86,NaN,NaN
2015-01-04,87.18,186.18,313.22,NaN,131.90,269.55,607.98,NaN,25.73,62.09,...,41.68,NaN,3.91,10.55,14.36,NaN,10.23,20.09,24.86,NaN
2015-01-05,151.84,87.18,186.18,NaN,241.84,131.90,269.55,NaN,25.01,25.73,...,29.97,NaN,4.26,3.91,10.55,NaN,9.71,10.23,20.09,NaN
2015-01-06,146.60,151.84,87.18,NaN,219.13,241.84,131.90,NaN,14.01,25.01,...,19.71,NaN,2.80,4.26,3.91,NaN,6.21,9.71,10.23,NaN
2015-01-07,149.58,146.60,151.84,NaN,252.10,219.13,241.84,NaN,17.21,14.01,...,25.36,NaN,3.63,2.80,4.26,NaN,7.35,6.21,9.71,NaN
2015-01-08,217.87,149.58,146.60,313.22,376.51,252.10,219.13,607.98,26.99,17.21,...,23.20,41.68,4.93,3.63,2.80,14.36,9.42,7.35,6.21,24.86
2015-01-09,229.90,217.87,149.58,186.18,360.95,376.51,252.10,269.55,23.34,26.99,...,26.83,29.97,5.80,4.93,3.63,10.55,11.40,9.42,7.35,20.09


In [11]:
# Verify lag direction on one date

check_date = pd.Timestamp("2017-01-10")

if check_date in processed_df.index:
    check = processed_df.loc[check_date]

    print("Date:", check_date.date())
    print("PM2.5 on current date:", check["PM2.5"])
    print("PM2.5 lag 1 (previous day):", check["PM2.5_lag_1"])
    print("PM2.5 lag 2:", check["PM2.5_lag_2"])
    print("PM2.5 lag 3:", check["PM2.5_lag_3"])
    print("PM2.5 lag 7:", check["PM2.5_lag_7"])
else:
    print("Check date not found.")

Date: 2017-01-10
PM2.5 on current date: 108.26
PM2.5 lag 1 (previous day): 141.88
PM2.5 lag 2: 136.98
PM2.5 lag 3: 154.52
PM2.5 lag 7: 255.33


In [12]:
print("Index type:", type(processed_df.index).__name__)
print("Index sorted:", processed_df.index.is_monotonic_increasing)
print("Duplicate dates:", processed_df.index.duplicated().sum())

date_gaps = processed_df.index.to_series().diff().dropna()
print("Non-daily intervals:", (date_gaps != pd.Timedelta(days=1)).sum())

print("Number of lag columns:", len(lag_columns))
print("Missing values in lag columns:", processed_df[lag_columns].isna().sum().sum())

Index type: DatetimeIndex
Index sorted: True
Duplicate dates: 0
Non-daily intervals: 0
Number of lag columns: 44
Missing values in lag columns: 1183


## 8. Historical Rolling Features — Ishan (Week 3)

Create rolling mean and standard deviation features over the **previous 3 and 7 calendar days** for each available pollutant. Each window ends at **t−1**, so the current day's value and future readings are excluded. A full window is required; missing readings produce `NaN` rather than silently changing the window size. These features are added to the existing `processed_df` without overwriting the original pollutant columns or Deepanshu's lag features.

In [13]:
# Ishan: rolling statistics based only on previous days
# processed_df is created in Abhishekk's section and extended by Deepanshu.
assert isinstance(processed_df.index, pd.DatetimeIndex)
assert processed_df.index.is_monotonic_increasing
assert not processed_df.index.has_duplicates
assert (processed_df.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all()

rolling_pollutants = [
    c for c in ["PM2.5", "PM10", "NO", "NO2", "NOx", "NH3", "CO", "SO2", "O3", "Benzene", "Toluene"]
    if c in processed_df.columns
]
rolling_columns = []

for pollutant in rolling_pollutants:
    history = processed_df[pollutant].shift(1)
    for window in (3, 7):
        mean_col = f"{pollutant}_roll_mean_{window}"
        std_col = f"{pollutant}_roll_std_{window}"
        processed_df[mean_col] = history.rolling(window=window, min_periods=window).mean()
        processed_df[std_col] = history.rolling(window=window, min_periods=window).std()
        rolling_columns.extend([mean_col, std_col])

print(
    "Rolling features preserved:",
    all(col in processed_df.columns for col in rolling_columns)
)

print(
    "Dataset shape:",
    processed_df.shape
)
print(f"Created {len(rolling_columns)} historical rolling features.")
display(processed_df[rolling_columns].head(10))

Rolling features preserved: True
Dataset shape: (2009, 102)
Created 44 historical rolling features.


,PM2.5_roll_mean_3,PM2.5_roll_std_3,PM2.5_roll_mean_7,PM2.5_roll_std_7,PM10_roll_mean_3,PM10_roll_std_3,PM10_roll_mean_7,PM10_roll_std_7,NO_roll_mean_3,NO_roll_std_3,...,O3_roll_mean_7,O3_roll_std_7,Benzene_roll_mean_3,Benzene_roll_std_3,Benzene_roll_mean_7,Benzene_roll_std_7,Toluene_roll_mean_3,Toluene_roll_std_3,Toluene_roll_mean_7,Toluene_roll_std_7
Date,,,,,,,,,,,,,,,,,,,,,
2015-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2015-01-04,195.526667,113.309490,NaN,NaN,336.476667,244.994746,NaN,NaN,52.326667,23.303073,...,NaN,NaN,9.606667,5.288481,NaN,NaN,18.393333,7.461115,NaN,NaN
2015-01-05,141.733333,50.267868,NaN,NaN,214.430000,72.803583,NaN,NaN,37.610000,21.203358,...,NaN,NaN,6.240000,3.736670,NaN,NaN,13.343333,5.848567,NaN,NaN
2015-01-06,128.540000,35.914504,NaN,NaN,197.623333,58.039671,NaN,NaN,21.583333,6.568572,...,NaN,NaN,3.656667,0.762255,NaN,NaN,8.716667,2.186352,NaN,NaN
2015-01-07,149.340000,2.628231,NaN,NaN,237.690000,16.872229,NaN,NaN,18.743333,5.658033,...,NaN,NaN,3.563333,0.732280,NaN,NaN,7.756667,1.785086,NaN,NaN
2015-01-08,171.350000,40.315046,178.924286,71.485747,282.580000,82.999331,299.858571,153.897740,19.403333,6.762258,...,27.958571,6.976574,3.786667,1.073608,6.348571,4.361557,7.660000,1.627298,12.552857,7.057716
2015-01-09,199.116667,43.319640,167.021429,48.704398,329.853333,67.784320,264.568571,83.847823,22.513333,4.942128,...,26.362857,3.925208,4.786667,1.092078,5.125714,2.575182,9.390000,2.025167,10.630000,4.524927


### Verify one row and check for leakage

For the selected date, a 3-day window must use only **t−3, t−2, t−1**; a 7-day window must use **t−7 through t−1**. Pandas `.std()` uses the sample standard deviation (`ddof=1`).

In [14]:
check_date = pd.Timestamp("2017-01-10")
if check_date in processed_df.index:
    for window in (3, 7):
        previous_days = pd.date_range(end=check_date - pd.Timedelta(days=1), periods=window)
        history_values = processed_df.loc[previous_days, "PM2.5"]
        expected_mean = history_values.mean() if history_values.notna().all() else np.nan
        expected_std = history_values.std() if history_values.notna().all() else np.nan
        actual_mean = processed_df.at[check_date, f"PM2.5_roll_mean_{window}"]
        actual_std = processed_df.at[check_date, f"PM2.5_roll_std_{window}"]
        assert np.isclose(actual_mean, expected_mean, equal_nan=True)
        assert np.isclose(actual_std, expected_std, equal_nan=True)
        print(f"{window}-day window for {check_date.date()}:")
        print(history_values.to_string())
        print(f"Verified mean: {actual_mean:.3f}, std: {actual_std:.3f}\n")
else:
    print("Selected verification date is not present.")

# Changing today's measurement cannot change today's historical rolling features.
probe = processed_df["PM2.5"].copy()
probe.iloc[-1] = -99999
original_today = processed_df["PM2.5"].shift(1).rolling(3, min_periods=3).mean().iloc[-1]
changed_today = probe.shift(1).rolling(3, min_periods=3).mean().iloc[-1]
assert np.isclose(original_today, changed_today, equal_nan=True)
print("Leakage check passed: current-day PM2.5 does not affect current-day historical rolling mean.")

3-day window for 2017-01-10:
2017-01-07    154.52
2017-01-08    136.98
2017-01-09    141.88
Freq: D
Verified mean: 144.460, std: 9.050

7-day window for 2017-01-10:
2017-01-03    255.33
2017-01-04    248.01
2017-01-05    215.42
2017-01-06    194.12
2017-01-07    154.52
2017-01-08    136.98
2017-01-09    141.88
Freq: D
Verified mean: 192.323, std: 49.403

Leakage check passed: current-day PM2.5 does not affect current-day historical rolling mean.


### Week 3 notes — Ishan

- Rolling windows are calculated on the continuous, sorted daily calendar.
- `.shift(1)` ensures that the most recent reading in each rolling window is **yesterday (t−1)**.
- `rolling(3)` and `rolling(7)` compute means and sample standard deviations using complete windows.
- Missing historical values are left as missing; feature imputation, if needed, must be fitted using training data only after the chronological split.
- This section does not create next-day targets or alter the team's missing-data policy.

## Calendar Feature Engineering(Atishey Jain)

In this section, we create calendar-based features to help the model understand seasonal patterns in Delhi's air pollution. These features include the month, day of the week, and cyclic day-of-year values.

Since our goal is to predict tomorrow's pollution, we use the next day's calendar information. This is safe because tomorrow's date is already known and does not introduce data leakage.

In [15]:
delhi_df

,City,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
Date,,,,,,,,,,,,,,,
2015-01-01,Delhi,313.22,607.98,69.16,36.39,110.59,33.85,15.20,9.25,41.68,14.36,24.86,9.84,472.0,Severe
2015-01-02,Delhi,186.18,269.55,62.09,32.87,88.14,31.83,9.54,6.65,29.97,10.55,20.09,4.29,454.0,Severe
2015-01-03,Delhi,87.18,131.90,25.73,30.31,47.95,69.55,10.61,2.65,19.71,3.91,10.23,1.99,143.0,Moderate
2015-01-04,Delhi,151.84,241.84,25.01,36.91,48.62,130.36,11.54,4.63,25.36,4.26,9.71,3.34,319.0,Very Poor
2015-01-05,Delhi,146.60,219.13,14.01,34.92,38.25,122.88,9.20,3.33,23.20,2.80,6.21,2.96,325.0,Very Poor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-06-27,Delhi,39.80,155.94,10.88,21.46,22.47,31.43,0.87,10.38,18.88,1.69,19.99,0.43,112.0,Moderate
2020-06-28,Delhi,59.52,308.65,12.67,21.60,23.86,29.27,0.94,10.70,18.05,1.71,25.13,1.74,196.0,Moderate
2020-06-29,Delhi,44.86,184.12,10.50,21.57,21.94,27.97,0.88,11.58,26.61,2.13,23.80,1.13,233.0,Poor


In [16]:
calendar_df = delhi_df.copy()
forecast_dates = calendar_df.index + pd.Timedelta(days=1)

In [17]:
calendar_df["month"] = forecast_dates.month
calendar_df["day_of_week"] = forecast_dates.dayofweek
calendar_df["day_of_year"] = forecast_dates.dayofyear

In [18]:
calendar_df["day_of_year_sin"] = np.sin(
    2 * np.pi * calendar_df["day_of_year"] / 365.25
)

calendar_df["day_of_year_cos"] = np.cos(
    2 * np.pi * calendar_df["day_of_year"] / 365.25
)

calendar_df[
    [
        "month","day_of_week","day_of_year","day_of_year_sin","day_of_year_cos"
    ]
].head()

,month,day_of_week,day_of_year,day_of_year_sin,day_of_year_cos
Date,,,,,
2015-01-01,1,4,2,0.034398,0.999408
2015-01-02,1,5,3,0.051584,0.998669
2015-01-03,1,6,4,0.068755,0.997634
2015-01-04,1,0,5,0.085906,0.996303
2015-01-05,1,1,6,0.103031,0.994678


In [19]:
calendar_df.loc["2020-01-10",["month","day_of_week","day_of_year","day_of_year_sin","day_of_year_cos"]]

month               1.000000
day_of_week         5.000000
day_of_year        11.000000
day_of_year_sin     0.188099
day_of_year_cos     0.982150
Name: 2020-01-10 00:00:00, dtype: float64

### Calendar Feature Engineering — Observations

We created calendar features such as month, day of week, and day of year to help the model identify seasonal patterns in Delhi's air pollution.

Since we are predicting the next day's pollution, the calendar features are calculated using tomorrow's date. This does not cause data leakage because future calendar dates are already known.

We also used sine and cosine transformations for day of year. This helps the model understand that the end of December and the beginning of January are close to each other in the yearly cycle.

We verified the features using 10 January 2020. The calculated values correctly correspond to 11 January 2020.

In [20]:
calendar_columns = [
    "month",
    "day_of_week",
    "day_of_year",
    "day_of_year_sin",
    "day_of_year_cos"
]

In [21]:
print("\nTotal calendar features:", len(calendar_columns))

print("Rolling features preserved:",all(col in processed_df.columns for col in rolling_columns))

print("Dataset shape:",processed_df.shape)


Total calendar features: 5
Rolling features preserved: True
Dataset shape: (2009, 102)


## 10. Next-Day Target Creation and Leakage Validation — Sahil (Week 3)

For each date **t**, create the two supervised-learning targets from the observation on the next calendar day **t+1**:

- `PM2.5_tomorrow`: next-day PM2.5 concentration (regression target)
- `AQI_Bucket_tomorrow`: next-day AQI category (classification target)

Because the Delhi data were reindexed to a continuous daily calendar earlier in this notebook, `.shift(-1)` aligns the next calendar day's observed outcome with the current row. The targets are labels only and must not be used as predictor features.


In [22]:
# Sahil: create next-day targets
processed_df = processed_df.sort_index().copy()

processed_df["PM2.5_tomorrow"] = processed_df["PM2.5"].shift(-1)
processed_df["AQI_Bucket_tomorrow"] = processed_df["AQI_Bucket"].shift(-1)

print("Created targets: PM2.5_tomorrow and AQI_Bucket_tomorrow")
print("Dataset shape after adding targets:", processed_df.shape)
processed_df[["PM2.5", "PM2.5_tomorrow", "AQI_Bucket", "AQI_Bucket_tomorrow"]].head(10)


Created targets: PM2.5_tomorrow and AQI_Bucket_tomorrow
Dataset shape after adding targets: (2009, 104)


,PM2.5,PM2.5_tomorrow,AQI_Bucket,AQI_Bucket_tomorrow
Date,,,,
2015-01-01,313.22,186.18,Severe,Severe
2015-01-02,186.18,87.18,Severe,Moderate
2015-01-03,87.18,151.84,Moderate,Very Poor
2015-01-04,151.84,146.60,Very Poor,Very Poor
2015-01-05,146.60,149.58,Very Poor,Very Poor
2015-01-06,149.58,217.87,Very Poor,Very Poor
2015-01-07,217.87,229.90,Very Poor,Very Poor
2015-01-08,229.90,201.66,Very Poor,Very Poor
2015-01-09,201.66,221.02,Very Poor,Very Poor


### Manual shift-direction check

The following check uses a real date from the Delhi series. For date **t**, the target stored on that row must equal the original observed value on **t+1**. This verifies that `shift(-1)` points forward by one day rather than backward.


In [23]:
# Sahil: manually verify target alignment on a real date
valid_dates = processed_df.index[:-1]
valid_dates = valid_dates[
    processed_df.loc[valid_dates, "PM2.5"].notna().to_numpy()
    & processed_df.loc[valid_dates + pd.Timedelta(days=1), "PM2.5"].notna().to_numpy()
]

check_date = pd.Timestamp("2020-01-10")
if check_date not in valid_dates:
    check_date = valid_dates[0]
next_date = check_date + pd.Timedelta(days=1)

print("Current date:", check_date.date())
print("Next date:   ", next_date.date())
print("Current PM2.5:", processed_df.loc[check_date, "PM2.5"])
print("Stored PM2.5_tomorrow:", processed_df.loc[check_date, "PM2.5_tomorrow"])
print("Actual next-day PM2.5: ", processed_df.loc[next_date, "PM2.5"])
print("Stored AQI_Bucket_tomorrow:", processed_df.loc[check_date, "AQI_Bucket_tomorrow"])
print("Actual next-day AQI_Bucket: ", processed_df.loc[next_date, "AQI_Bucket"])

assert processed_df.loc[check_date, "PM2.5_tomorrow"] == processed_df.loc[next_date, "PM2.5"]

stored_bucket = processed_df.loc[check_date, "AQI_Bucket_tomorrow"]
actual_bucket = processed_df.loc[next_date, "AQI_Bucket"]
assert (pd.isna(stored_bucket) and pd.isna(actual_bucket)) or stored_bucket == actual_bucket

print("\nPASS: both targets are aligned with t+1.")


Current date: 2020-01-10
Next date:    2020-01-11
Current PM2.5: 130.82
Stored PM2.5_tomorrow: 151.1
Actual next-day PM2.5:  151.1
Stored AQI_Bucket_tomorrow: Very Poor
Actual next-day AQI_Bucket:  Very Poor

PASS: both targets are aligned with t+1.


### Full alignment and leakage checks

We now validate the target construction across the complete series. Predictor columns must not include either target column. Missing target labels are kept as missing rather than imputed, because inventing a target would change the observed outcome.


In [24]:
# Sahil: full-series alignment checks
expected_pm25_target = processed_df["PM2.5"].shift(-1)
expected_bucket_target = processed_df["AQI_Bucket"].shift(-1)

pm25_alignment_ok = processed_df["PM2.5_tomorrow"].equals(expected_pm25_target)
bucket_alignment_ok = processed_df["AQI_Bucket_tomorrow"].equals(expected_bucket_target)

assert pm25_alignment_ok, "PM2.5 target alignment failed"
assert bucket_alignment_ok, "AQI_Bucket target alignment failed"

target_columns = ["PM2.5_tomorrow", "AQI_Bucket_tomorrow"]
predictor_columns = [col for col in processed_df.columns if col not in target_columns]
assert not any(col in predictor_columns for col in target_columns)

print("PM2.5 t+1 alignment:", pm25_alignment_ok)
print("AQI_Bucket t+1 alignment:", bucket_alignment_ok)
print("Target columns excluded from predictor list:", all(c not in predictor_columns for c in target_columns))
print("Missing PM2.5 target labels:", processed_df["PM2.5_tomorrow"].isna().sum())
print("Missing AQI_Bucket target labels:", processed_df["AQI_Bucket_tomorrow"].isna().sum())


PM2.5 t+1 alignment: True
AQI_Bucket t+1 alignment: True
Target columns excluded from predictor list: True
Missing PM2.5 target labels: 3
Missing AQI_Bucket target labels: 11


### Target missingness policy

Target values are **not filled or imputed**. Rows without an observed next-day label should be excluded only when preparing the corresponding supervised-learning dataset. This avoids fabricating outcomes and keeps feature preprocessing separate from target handling.


In [25]:
# Build task-specific modeling views without modifying/filling target labels
regression_df = processed_df.dropna(subset=["PM2.5_tomorrow"]).copy()
classification_df = processed_df.dropna(subset=["AQI_Bucket_tomorrow"]).copy()

print("Rows available for PM2.5 regression:", len(regression_df))
print("Rows available for AQI_Bucket classification:", len(classification_df))
print("Original processed rows preserved:", len(processed_df))


Rows available for PM2.5 regression: 2006
Rows available for AQI_Bucket classification: 1998
Original processed rows preserved: 2009


### Sahil — Week 3 validation summary

- Created next-day PM2.5 and AQI category targets using `.shift(-1)`.
- Manually checked a real date against the following calendar day's observed values.
- Added full-series assertions to verify target alignment.
- Explicitly excluded target columns from the predictor-column list to prevent target leakage.
- Kept missing target labels unfilled; task-specific datasets drop rows only when that task's target is unavailable.
- The original chronological ordering is preserved for later time-ordered train/test splitting.
